In [ ]:
import cv2
import numpy as np

import cv2
import numpy as np


def preprocess_gray(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    return gray


def bandpass_small_objects(gray):
    # emphasize small blob-like structures
    g1 = cv2.GaussianBlur(gray, (0, 0), 1.0)
    g2 = cv2.GaussianBlur(gray, (0, 0), 3.5)
    dog = cv2.absdiff(g1, g2)
    _, dog_mask = cv2.threshold(dog, 10, 255, cv2.THRESH_BINARY)
    return dog_mask


def motion_map_fast_ball(prev_frame, curr_frame, next_frame, roi_mask=None, thr=14):
    p = preprocess_gray(prev_frame)
    c = preprocess_gray(curr_frame)
    n = preprocess_gray(next_frame)

    d1 = cv2.absdiff(c, p)
    d2 = cv2.absdiff(n, c)

    _, m1 = cv2.threshold(d1, thr, 255, cv2.THRESH_BINARY)
    _, m2 = cv2.threshold(d2, thr, 255, cv2.THRESH_BINARY)

    # IMPORTANT: shift-tolerant overlap
    k = np.ones((5, 5), np.uint8)
    m1d = cv2.dilate(m1, k, iterations=1)
    m2d = cv2.dilate(m2, k, iterations=1)
    persistent = cv2.bitwise_and(m1d, m2d)

    # small-object emphasis
    dog_mask = bandpass_small_objects(c)

    # combine motion + small object prior
    motion = cv2.bitwise_and(persistent, dog_mask)

    if roi_mask is not None:
        motion = cv2.bitwise_and(motion, roi_mask)

    # remove isolated noise without growing blobs too much
    motion = cv2.morphologyEx(
        motion, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8)
    )

    return motion


In [ ]:
def extract_ball_candidates(
    motion,
    curr_frame,
    predicted_xy=None,
    gate_radius=120,
    min_area=3,
    max_area=35,
):
    gray = preprocess_gray(curr_frame)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(motion, connectivity=8)

    candidates = []

    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        cx, cy = centroids[i]

        if area < min_area or area > max_area:
            continue

        # ball may be slightly elongated due to blur, but not too much
        aspect = w / max(h, 1)
        if aspect < 0.5 or aspect > 2.2:
            continue

        # compactness proxy
        box_area = w * h
        fill_ratio = area / max(box_area, 1)
        if fill_ratio < 0.2:
            continue

        # local contrast
        x0 = max(0, x - 2)
        y0 = max(0, y - 2)
        x1 = min(gray.shape[1], x + w + 2)
        y1 = min(gray.shape[0], y + h + 2)
        patch = gray[y0:y1, x0:x1]
        if patch.size == 0:
            continue

        contrast = int(patch.max()) - int(patch.min())
        if contrast < 15:
            continue

        # score: SMALLER is better than bigger
        score = 0.0
        score += 40 - min(area, 40)          # reward small blobs
        score += contrast * 0.8
        score += fill_ratio * 30

        if predicted_xy is not None:
            dist = np.hypot(cx - predicted_xy[0], cy - predicted_xy[1])
            if dist > gate_radius:
                continue
            score += max(0, gate_radius - dist) * 0.8

        candidates.append({
            "bbox": (int(x), int(y), int(w), int(h)),
            "center": (float(cx), float(cy)),
            "area": int(area),
            "aspect": float(aspect),
            "fill_ratio": float(fill_ratio),
            "contrast": float(contrast),
            "score": float(score),
        })

    candidates.sort(key=lambda c: c["score"], reverse=True)
    return candidates

In [4]:
def generate_ball_candidates(prev_frame, curr_frame, next_frame, roi_mask=None, predicted_xy=None):
    motion = motion_map(prev_frame, curr_frame, next_frame, roi_mask=roi_mask, thr=12)
    candidates = extract_candidates(
        motion,
        curr_frame,
        min_area=4,
        max_area=120,
        predicted_xy=predicted_xy,
        gate_radius=140,
    )
    return motion, candidates


In [ ]:
def make_roi_mask(frame_shape):
    h, w = frame_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    # broad playable region + air above court
    pts = np.array([
        [80, 80],
        [1200, 80],
        [1160, 610],
        [120, 610]
    ], dtype=np.int32)

    cv2.fillPoly(mask, [pts], 255)

    # remove scoreboard strip
    cv2.rectangle(mask, (0, 590), (w, h), 0, -1)

    # remove watermark corner
    cv2.rectangle(mask, (1020, 560), (w, h), 0, -1)

    return mask

In [6]:
from pathlib import Path
import json

frames_dir = Path('/home/bugslayer/Downloads/volley video footage/frames/videoplayback-00.06.05.191-00.11.14.198')
annotated_dir = Path('/home/bugslayer/Downloads/volley video footage/annotated_frames')
candidates_dir = Path('/home/bugslayer/Downloads/volley video footage/candidates')

annotated_dir.mkdir(parents=True, exist_ok=True)
candidates_dir.mkdir(parents=True, exist_ok=True)

frame_paths = sorted(frames_dir.glob('*.jpg'))
if len(frame_paths) < 3:
    raise ValueError(f'Expected at least 3 frames in {frames_dir}, got {len(frame_paths)}')

first_frame = cv2.imread(str(frame_paths[0]))
if first_frame is None:
    raise RuntimeError(f'Failed to read first frame: {frame_paths[0]}')

roi_mask = make_roi_mask(first_frame.shape)
MAX_CANDIDATES_TO_DRAW = 6
max_frames = len(frame_paths)

def draw_candidates(frame, candidates):
    vis = frame.copy()
    for i, c in enumerate(candidates[:MAX_CANDIDATES_TO_DRAW]):
        x, y, w, h = c['bbox']
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 255), 2)
        label = f'cand-{i}: s={c["score"]:.1f} a={c["area"]}'
        cv2.putText(
            vis,
            label,
            (x, max(0, y - 6)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.35,
            (0, 255, 255),
            1,
        )

    return vis

predicted_xy = None
processed = 0

for idx in range(1, max_frames - 1):
    prev = cv2.imread(str(frame_paths[idx - 1]))
    curr = cv2.imread(str(frame_paths[idx]))
    nxt = cv2.imread(str(frame_paths[idx + 1]))
    if prev is None or curr is None or nxt is None:
        continue

    _, candidates = generate_ball_candidates(
        prev,
        curr,
        nxt,
        roi_mask=roi_mask,
        predicted_xy=predicted_xy,
    )

    vis = draw_candidates(curr, candidates)
    cv2.imwrite(str(annotated_dir / frame_paths[idx].name), vis)

    payload = {
        'frame': frame_paths[idx].name,
        'index': idx,
        'num_candidates': len(candidates),
        'candidates': candidates,
    }
    (candidates_dir / f'{frame_paths[idx].stem}.json').write_text(
        json.dumps(payload, indent=2),
        encoding='utf-8',
    )

    if candidates:
        predicted_xy = candidates[0]['center']

    processed += 1

print(f'Processed {processed} frames with custom candidate functions')


KeyboardInterrupt: 